In [ ]:
HUGGINGFACEHUB_API_TOKEN = "Your_Api_Token"

import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate

# llm_name = "meta-llama/Llama-3.1-8B"  # Text generation model
llm_name = "meta-llama/Llama-3.1-8B-Instruct"  # Chat/Instruct model

# Initialize the endpoint -> model running on remote HuggingFace's servers
llm = HuggingFaceEndpoint(
    repo_id=llm_name,
    task="text-generation",
    # max_new_tokens=100
)

# Wrap HuggingFaceEndpoint object llm; extract specific format required by original chat model llm_name
model = ChatHuggingFace(llm=llm)
# With "model" rather than "llm", plain string prompt/message will be formatted before being sent to llm_name

input_text = "Explain this concept simply and concisely: {concept}"
prompt_template = PromptTemplate.from_template(input_text)

chain = prompt_template | model  # Create a chain using LangChain Expression Language (LCEL)

concept = "AI Agent"
response = chain.invoke({"concept": concept})
print(f"1. What's {concept}?")
print(response.content)

# Initialize chat model llm_name with init_chat_model() -> ERROR: Cannot access gated repo! WHY?
# from langchain.chat_models import init_chat_model

# model2 = init_chat_model(
#     llm_name,
#     model_provider="huggingface",
#     huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
# )

from langchain.agents import create_agent
from langchain.tools import tool
import numexpr as ne

@tool
def calculator(expression: str) -> str:
    """Performs arithmetic calculations. Use this for any math problems."""
    # Note: Use a safe evaluation library like 'numexpr' for production
    return str(ne.evaluate(expression))

# Create an agent with create_agent(); available since v1.0   # model = llm_name -> ERROR!
agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="You are a helpful assistant",
)

question = "What's the square root of 99?"
message = agent.invoke({"messages": [{"role": "user", "content": question}]})
print("\n2. " + question)
print(message["messages"][-1].content)

1. What's AI Agent?
An **AI Agent** is a smart software program that uses artificial intelligence (AI) to perform tasks, make decisions, or solve problems on its own or with minimal human input. It can learn from data, adapt to situations, and act in environments like a digital assistant, chatbot, or automated system.

**Key Features:**
- **Autonomy** – Works independently.
- **Learning** – Improves over time using AI models.
- **Decision-Making** – Chooses actions based on data and goals.
- **Interaction** – Can communicate with users or other systems.

**Examples:**
- **Chatbots** (e.g., customer service assistants)
- **Recommendation Systems** (e.g., Netflix, Spotify)
- **Autonomous Vehicles** (self-driving cars)

In short, an AI Agent is a smart, self-operating program that helps automate tasks using AI.

2. What's the square root of 99?
The square root of 99 is approximately 9.9498743710662.
